# Preprocessing Pipeline

**Objective:** Implement and test the video preprocessing pipeline including frame extraction, face detection, and image normalization.

**Research Traceability:**
- Research Objective: Frame-level preprocessing for deepfake detection
- Methodology: OpenCV-based frame extraction and MTCNN face detection
- Implementation: notebooks/02_preprocessing.ipynb

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Frame Extraction

In [ ]:
def extract_frames(video_path: str, num_frames: int = 10) -> list[np.ndarray]:
    """Extract evenly spaced frames from video."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        return []
    
    # Calculate frame indices
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    cap.release()
    return frames

# Test on a sample video
sample_video = '../datasets/raw/faceforensics/original/000.mp4'
if Path(sample_video).exists():
    frames = extract_frames(sample_video, num_frames=5)
    print(f"Extracted {len(frames)} frames")
    
    # Display frames
    fig, axes = plt.subplots(1, len(frames), figsize=(20, 4))
    for i, frame in enumerate(frames):
        axes[i].imshow(frame)
        axes[i].set_title(f'Frame {i+1}')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Sample video not found")

## 2. Face Detection

In [ ]:
from src.preprocessing.face_detector import FaceDetector

# Initialize face detector
detector = FaceDetector(method='mtcnn', device='cpu')

def detect_faces_demo(frame: np.ndarray) -> np.ndarray:
    """Detect faces and draw bounding boxes."""
    frame_copy = frame.copy()
    detections = detector.detect(frame)
    
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        confidence = det['confidence']
        
        # Draw bounding box
        cv2.rectangle(frame_copy, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame_copy, f'{confidence:.2f}', (x1, y1 - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    return frame_copy

# Test face detection
if 'frames' in locals() and len(frames) > 0:
    detected_frame = detect_faces_demo(frames[0])
    
    plt.figure(figsize=(10, 8))
    plt.imshow(detected_frame)
    plt.title('Face Detection Results')
    plt.axis('off')
    plt.show()
else:
    print("No frames available for testing")

## 3. Face Cropping and Normalization

In [ ]:
from src.preprocessing.face_cropper import FaceCropper
from src.preprocessing.image_resizer import ImageResizer
from src.preprocessing.normalizer import ImageNormalizer

# Initialize components
cropper = FaceCropper(output_size=299, margin=0.2)
resizer = ImageResizer(target_size=299)
normalizer = ImageNormalizer()

def preprocess_frame(frame: np.ndarray, detections: list) -> list[np.ndarray]:
    """Preprocess detected faces."""
    processed = []
    
    for det in detections:
        # Crop face
        face = cropper.crop(frame, det['bbox'])
        
        # Resize
        face = resizer.resize(face)
        
        # Normalize
        face = normalizer.normalize(face)
        
        processed.append(face)
    
    return processed

# Test preprocessing pipeline
if 'frames' in locals() and len(frames) > 0:
    detections = detector.detect(frames[0])
    processed_faces = preprocess_frame(frames[0], detections)
    
    print(f"Detected {len(detections)} faces")
    print(f"Processed {len(processed_faces)} faces")
    
    # Display processed faces
    if processed_faces:
        fig, axes = plt.subplots(1, min(3, len(processed_faces)), figsize=(12, 4))
        for i, face in enumerate(processed_faces[:3]):
            axes[i].imshow(face)
            axes[i].set_title(f'Face {i+1}')
            axes[i].axis('off')
        plt.tight_layout()
        plt.show()

## 4. Preprocessing Statistics

## Summary
- Frame extraction with configurable frame count
- Face detection using MTCNN with confidence thresholding
- Face cropping with margin padding
- Image resizing to target dimensions (299x299 for XceptionNet)
- Normalization to [0, 1] range

## Next Steps
- Train models: `04_training_xception.ipynb` or `05_training_efficientnet.ipynb`